In [ ]:
import pandas as pd
import numpy as np
import os
import nepali_datetime

INPUT_PATH = r'C:\Users\a2z\OneDrive\Desktop\solar_power_prediction\data\raw\solar_power_data.xlsx'
OUTPUT_PATH = r'C:\Users\a2z\OneDrive\Desktop\solar_power_prediction\data\processed\cleaned_solar_data.csv'

INSTALLED_CAPACITY_KW = 1000  

all_sheets = pd.read_excel(INPUT_PATH, sheet_name=None)
df = pd.concat(all_sheets.values(), ignore_index=True)

df.columns = [col.strip() for col in df.columns]

def convert_date(bs_val):
    try:
        if isinstance(bs_val, pd.Timestamp) or hasattr(bs_val, 'year'):
            y, m, d = bs_val.year, bs_val.month, bs_val.day
        else:
            y, m, d = map(int, str(bs_val).split('-'))
        
        return nepali_datetime.date(y, m, d).to_datetime_date()
    except Exception as e:
        return None

df['Date_AD'] = df['Date'].apply(convert_date)

df = df.dropna(subset=['Date_AD'])
df['Date_AD'] = pd.to_datetime(df['Date_AD'])

df = df.sort_values('Date_AD').reset_index(drop=True)

# ENGINEERING METRIC CALCULATIONS
# Specific Yield (kWh/kWp)
df['Specific_Yield'] = df['Generated Energy (KWH)'] / INSTALLED_CAPACITY_KW

# Capacity Utilization Factor (CUF %)
# Formula: (Energy / (Capacity * 24)) * 100
df['CUF_Percentage'] = (df['Generated Energy (KWH)'] / (INSTALLED_CAPACITY_KW * 24)) * 100

df['Month'] = df['Date_AD'].dt.month
df['Day_of_Year'] = df['Date_AD'].dt.dayofyear
df['Day_of_Week'] = df['Date_AD'].dt.dayofweek

# SAVE DATA
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Success! Processed {len(df)} rows.")
print(df[['Date', 'Date_AD', 'CUF_Percentage']].head())

Success! Processed 149 rows.
                  Date    Date_AD  CUF_Percentage
0  2081-02-01 00:00:00 2024-05-14       62.087500
1  2081-02-02 00:00:00 2024-05-15       94.933333
2  2081-02-03 00:00:00 2024-05-16       94.312500
3  2081-02-04 00:00:00 2024-05-17       82.008333
4  2081-02-05 00:00:00 2024-05-18       80.841667
